# SQL Analysis

This notebook performs SQL-based analysis on the cleaned movie datasets stored in MySQL.

The analysis includes:
- Retrieving data using SQL queries
- Filtering and sorting records
- Aggregating data using `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX`
- Grouping data using `GROUP BY`
- Filtering grouped results using `HAVING`
- Combining related tables using `JOIN`
- Extracting meaningful insights from the data

In [2]:
import pandas as pd
import pymysql

In [ ]:
connection = pymysql.connect(
    host="localhost",
    user="****",
    password="*****",
    database="TMBD_analytics"
)

*List all movies in the Action genre released after 2015, showing title and release date.*

In [4]:
query = """
SELECT m.title, m.release_date
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action'
  AND YEAR(m.release_date) > 2015;
"""

result = pd.read_sql(query, connection)
result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\57717964.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, connection)


,title,release_date
0,Deadpool,2016-02-09
1,Avengers: Infinity War,2018-04-25
2,Avengers: Endgame,2019-04-24
3,Captain America: Civil War,2016-04-27
4,Black Panther,2018-02-13
...,...,...
263,Peppermint,2018-09-06
264,Mortal Kombat II,2026-05-06
265,Day Shift,2022-08-05
266,Samaritan,2022-08-25


*Find the top 10 highest-grossing movies along with the genre(s) each belongs to.*

In [5]:
query = """
SELECT m.title, m.revenue, g.genre_name
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
ORDER BY m.revenue DESC
LIMIT 10;
"""

result = pd.read_sql(query, connection)
result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\774748478.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, connection)


,title,revenue,genre_name
0,Avatar,2.923710e+09,Action
1,Avatar,2.923710e+09,Science Fiction
2,Avatar,2.923710e+09,Adventure
3,Avengers: Endgame,2.799440e+09,Adventure
4,Avengers: Endgame,2.799440e+09,Science Fiction
5,Avengers: Endgame,2.799440e+09,Action
6,Avatar: The Way of Water,2.334480e+09,Action
7,Avatar: The Way of Water,2.334480e+09,Adventure
8,Avatar: The Way of Water,2.334480e+09,Science Fiction
9,Titanic,2.264160e+09,Drama


*Calculate the average budget and average revenue for each genre, sorted by average revenue descending.*

In [6]:
query = """
SELECT 
    g.genre_name,
    AVG(m.budget) AS average_budget,
    AVG(m.revenue) AS average_revenue
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY average_revenue DESC;
"""

result = pd.read_sql(query, connection)
result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\3708762328.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result = pd.read_sql(query, connection)


,genre_name,average_budget,average_revenue
0,Adventure,1.093600e+08,3.902145e+08
1,Animation,8.336061e+07,3.597435e+08
2,Family,8.458426e+07,3.430968e+08
3,Science Fiction,9.525992e+07,3.235601e+08
4,Action,9.329457e+07,3.000547e+08
5,Fantasy,8.617784e+07,2.942975e+08
6,Comedy,5.256907e+07,2.113513e+08
7,Music,4.106939e+07,1.974526e+08
8,Romance,3.461113e+07,1.627896e+08
9,Thriller,4.938770e+07,1.605152e+08


#*List the top 10 actors who have appeared in the most movies in the dataset*

In [7]:
query = """
SELECT 
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM cast
GROUP BY actor_name
ORDER BY movie_count DESC
LIMIT 10;
"""

q4_result = pd.read_sql(query, connection)
q4_result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\2923355096.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q4_result = pd.read_sql(query, connection)


,actor_name,movie_count
0,Samuel L. Jackson,39
1,Robert De Niro,38
2,Brad Pitt,38
3,Johnny Depp,37
4,Tom Hanks,33
5,Willem Dafoe,32
6,Mark Wahlberg,32
7,Scarlett Johansson,32
8,Morgan Freeman,31
9,Tom Cruise,31


#*Find all directors who have directed more than 3 movies, along with their movie count*

In [8]:
query = """
SELECT 
    person_name AS director,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE job = 'Director'
GROUP BY person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC;
"""

q5_result = pd.read_sql(query, connection)
q5_result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\2250658147.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q5_result = pd.read_sql(query, connection)


,director,movie_count
0,Steven Spielberg,27
1,Ridley Scott,19
2,Tim Burton,19
3,Martin Scorsese,16
4,Michael Bay,15
...,...,...
212,Stephen Sommers,4
213,Tate Taylor,4
214,Tim Johnson,4
215,Tim Story,4


#*List the top 10 most frequently used keywords across all movies*

In [9]:
query = """
SELECT 
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM keywords
GROUP BY keyword_name
ORDER BY movie_count DESC
LIMIT 10;
"""

q6_result = pd.read_sql(query, connection)
q6_result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\332044287.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q6_result = pd.read_sql(query, connection)


,keyword_name,movie_count
0,based on novel or book,449
1,sequel,416
2,duringcreditsstinger,296
3,aftercreditsstinger,242
4,murder,176
5,based on true story,166
6,new york city,161
7,villain,160
8,3d animation,155
9,based on comic,154


#*Find movies where the budget was above the overall average budget but the revenue was below the overall average revenue*

In [10]:
query = """
SELECT 
    title,
    budget,
    revenue
FROM movies
WHERE budget > (
    SELECT AVG(budget)
    FROM movies
)
AND revenue < (
    SELECT AVG(revenue)
    FROM movies
);
"""

q7_result = pd.read_sql(query, connection)
q7_result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\2483732167.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q7_result = pd.read_sql(query, connection)


,title,budget,revenue
0,Jarhead,72000000.0,97076200.0
1,Mars Attacks!,70000000.0,101371000.0
2,Snatch,100000000.0,83557900.0
3,Sin City: A Dame to Kill For,65000000.0,39407600.0
4,Braveheart,72000000.0,213216000.0
...,...,...,...
312,One Battle After Another,175000000.0,209377000.0
313,Frankenstein,120000000.0,480678.0
314,28 Years Later,60000000.0,151309000.0
315,Predator: Badlands,105000000.0,184500000.0


#*List all actors who have appeared in a movie directed by a specific director*

In [11]:
query = """
SELECT DISTINCT
    c.actor_name
FROM cast c
JOIN crew cr
    ON c.movie_id = cr.movie_id
WHERE cr.person_name = 'Christopher Nolan'
  AND cr.job = 'Director'
ORDER BY c.actor_name;
"""

q8_result = pd.read_sql(query, connection)
q8_result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\2301913253.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q8_result = pd.read_sql(query, connection)


,actor_name
0,Aaron Eckhart
1,Al Pacino
2,Alon Aboutboul
3,Andy Serkis
4,Aneurin Barnard
...,...
79,Tom Hardy
80,Tom Wilkinson
81,Topher Grace
82,Wes Bentley


#*Find the genre with the highest average rating, considering only genres with at least 20 movies.*

In [12]:
query = """
SELECT 
    g.genre_name,
    AVG(m.vote_average) AS average_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies m
JOIN movie_genres mg
    ON m.movie_id = mg.movie_id
JOIN genres g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY average_rating DESC
LIMIT 1;
"""


#*List the top 10 movies with the highest number of keywords tagged to them*

In [13]:
query = """
SELECT 
    m.title,
    COUNT(DISTINCT k.keyword_id) AS keyword_count
FROM movies m
JOIN keywords k
    ON m.movie_id = k.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10;
"""

q10_result = pd.read_sql(query, connection)
q10_result

C:\Users\Gokul\AppData\Local\Temp\ipykernel_27016\3964528315.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  q10_result = pd.read_sql(query, connection)


,title,keyword_count
0,Smile 2,98
1,Taken 3,70
2,Silent Hill,63
3,Twelve Monkeys,57
4,War for the Planet of the Apes,54
5,The Shining,52
6,Dawn of the Planet of the Apes,51
7,How to Train Your Dragon,50
8,How to Train Your Dragon: The Hidden World,49
9,13 Hours: The Secret Soldiers of Benghazi,49
